# beat2body: run the three pose tools on Colab (README 4.3)

Runs `pose_mediapipe.py`, `pose_dwpose.py` and `pose_openpose.py` from `2d/scripts/`
on a few videos and shows the skeleton preview videos and basic numbers.

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`.

Steps:
1. Install packages.
2. Download the scripts and only the selected videos from the team repository (not the whole 2 GB).
3. Run the three tools, then look at the summary table and the previews.

Use `Runtime -> Run all`.

In [1]:
# 1. Check the GPU and install packages
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: DWPose and OpenPose will run on CPU (slow)"
!pip -q install mediapipe onnxruntime-gpu
!pip -q install --no-deps easy-dwpose
# If pip reports that numpy was changed: Runtime -> Restart session, then continue from the next cell.

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
# 2. Choose videos and download only those
VIDEOS = [
    "la_hiphop/gLH_sFM_c01_d16_mLH0_ch01.mp4",
    "house/gHO_sFM_c01_d19_mHO0_ch01.mp4",
    "break/gBR_sFM_c01_d04_mBR0_ch01.mp4",
]
REPO = "/content/beat2body"
SOURCE = "https://github.com/SAMMYBOOOOM/beat2body"
BRANCH = "main"

import os
if not os.path.exists(REPO):
    !git clone -q --depth 1 --filter=blob:none --sparse -b {BRANCH} {SOURCE} {REPO}
else:
    !cd {REPO} && git pull -q origin {BRANCH}
patterns = " ".join(["'/*'", "'!/2d/raw_video/**/*.mp4'"] + [f"'/2d/raw_video/{v}'" for v in VIDEOS])
!cd {REPO} && git sparse-checkout set --no-cone {patterns}
for v in VIDEOS:
    p = f"{REPO}/2d/raw_video/{v}"
    print("OK     " if os.path.exists(p) else "MISSING", v)
print("scripts:", sorted(os.listdir(f"{REPO}/2d/scripts")))

remote: Enumerating objects: 2, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 2 (delta 0), reused 2 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (2/2), 1.88 KiB | 1.88 MiB/s, done.
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 12 (delta 0), reused 7 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (12/12), 36.16 MiB | 14.36 MiB/s, done.
Updating files: 100% (24/24), done.
OK      la_hiphop/gLH_sFM_c01_d16_mLH0_ch01.mp4
OK      house/gHO_sFM_c01_d19_mHO0_ch01.mp4
OK      break/gBR_sFM_c01_d04_mBR0_ch01.mp4
scripts: ['colab_build_openpose.sh', 'pipeline.py', 'pose_dwpose.py', 'pose_mediapipe.py', 'pose_openpose.py']


In [ ]:
# 3a. MediaPipe (CPU)
# MediaPipe's GPU mode needs OpenGL, which Colab does not provide, so it runs on CPU here.
# Videos that already have output are skipped; add --overwrite to run them again.
%cd {REPO}
!python 2d/scripts/pose_mediapipe.py --device cpu --preview

/content/beat2body
I0000 00:00:1794376427.576383     879 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[mediapipe] downloading heavy model
[mediapipe] running on cpu
[mediapipe] 3 videos
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1794376434.998728     968 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1794376435.137254     968 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1794376435.916996     968 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMEN

In [ ]:
# 3b. DWPose (GPU)
%cd {REPO}
!python 2d/scripts/pose_dwpose.py --preview

In [ ]:
# 3c. OpenPose (GPU, official CMU build)
# The first run compiles OpenPose from source (about 25 minutes); later runs reuse the build.
!bash {REPO}/2d/scripts/colab_build_openpose.sh
%cd {REPO}
!python 2d/scripts/pose_openpose.py --backend official --preview

In [ ]:
# 4. Summary: speed, frames with a person, visible joints per tool and video
import glob
import numpy as np
import pandas as pd

THR = {"dwpose": 0.3, "mediapipe": 0.5, "openpose": 0.1}
rows = []
files_by_tool = [(t, f) for t in THR for f in sorted(glob.glob(f"{REPO}/2d/keypoints/*/*_{t}.npz"))]
for tool, f in files_by_tool:
    z = np.load(f)
    k = z["keypoints"]
    found = ~np.isnan(k[:, :, 0]).all(1)
    rows.append({
        "video": os.path.basename(f)[: -len(f"_{tool}.npz")],
        "tool": tool,
        "frames": len(k),
        "speed_fps": round(len(k) / float(z["processing_seconds"]), 1),
        "person_found_%": round(100 * found.mean(), 1),
        "visible_joints_of_17": round(float((k[:, :, 2] >= THR[tool]).sum(1).mean()), 1),
    })
df = pd.DataFrame(rows).sort_values(["video", "tool"])
df

In [ ]:
# 5. Preview videos (one per tool and video)
from base64 import b64encode
from IPython.display import HTML, display
for f in sorted(glob.glob(f"{REPO}/2d/keypoints/*/*_*.mp4")):
    data = b64encode(open(f, "rb").read()).decode()
    display(HTML(f"<h4>{os.path.basename(f)}</h4>"
                 f"<video width=720 controls><source src='data:video/mp4;base64,{data}' type='video/mp4'></video>"))